**13/08/2026** -- Inline descriptive statistics for results section *Deprivation and exposure to local and transported fire pollution*


In [1]:
library(dplyr)
library(readr)
library(tidyr)
library(stringr)
library(brms)
library(rstan)
library(cmdstanr)
library(sf)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Rcpp

Loading 'brms' package (version 2.22.0). Useful instructions
can be found by typing help('brms'). A more detailed introduction
to the package is available through vignette('brms_overview').


Attaching package: ‘brms’


The following object is masked from ‘package:stats’:

    ar


Loading required package: StanHeaders


rstan version 2.32.6 (Stan version 2.32.2)


For execution on a local, multicore CPU with excess RAM we recommend calling
options(mc.cores = parallel::detectCores()).
To avoid recompilation of unchanged Stan programs, we recommend calling
rstan_options(auto_write = TRUE)
For within-chain threading using `reduce_sum()` or `map_rect()` Stan functions,
change `threads_per_chain` option:
rstan_options(threads_per_chain = 1)



Attaching package

In [2]:
root <- rprojroot::find_root(rprojroot::has_file(".gitignore"))
source(file.path(root, "src/deprivation_pm_bhm/data_prep.R"))
source(file.path(root, "src/deprivation_pm_bhm/pca.R"))
source(file.path(root, "src/deprivation_pm_bhm/model_setup.R"))
source(file.path(root, "src/deprivation_pm_bhm/model_summary.R"))
source(file.path(root, "src/deprivation_pm_bhm/postprocess.R"))
source(file.path(root, "src/deprivation_pm_bhm/plotting.R"))
source(file.path(root, "src/utils/utils.R"))

In [3]:
SCRATCH_DIR <- Sys.getenv("SCRATCH_DIR")
NAT_BOUNDS_PATH     <- file.path(SCRATCH_DIR, 
                         "data/spatial/nat_boundaries",
                         "WB_countries_Admin0_10m")
western_sahara_path <- file.path(SCRATCH_DIR, 
                         "data/spatial/nat_boundaries",
                         "western_sahara/gadm41_ESH_0.shp")

In [4]:
nat_bounds <- read_sf(NAT_BOUNDS_PATH)
esh_bounds <- read_sf(western_sahara_path)

Warning message in CPL_read_ogr(dsn, layer, query, as.character(options), quiet, :
“GDAL Error 1: PROJ: proj_identify: Open of /home/users/cho00/miniconda3/envs/ppca/share/proj failed”
Warning message in CPL_read_ogr(dsn, layer, query, as.character(options), quiet, :
“GDAL Error 1: PROJ: proj_identify: Open of /home/users/cho00/miniconda3/envs/ppca/share/proj failed”


In [5]:
# Function to post-process model
postprocess <- function(model, df, nat_bounds) {
    draws   <- as_draws_df(model)

    # PC1 effects in U & R (draws)
    pc1_draws_ur <- extract_pc1_draws_ur(draws)      
    
    # Posterior summary by country & U/R
    slopes_ur <- summarise_draws(
        pc1_draws_ur, 
        pivot = TRUE, pivot_cols = c("rural", "urban"),
        names_to = "urban_rural_cat",
        group_vars = c("country", "urban_rural_cat", "term")
    )
    
    # Post-stratification: p-w avg of U & R PC1 draws
    pc1_draws_pw_avg <- compute_pw_pc1_draws(pc1_draws_ur, df)
    
    # Posterior summary of U/R pooled effects draws
    slopes_pw_avg <- summarise_draws( 
        pc1_draws_pw_avg, 
        pivot = TRUE, pivot_cols = c("pooled"),
        names_to = "urban_rural_cat",
        group_vars = c("country", "urban_rural_cat", "term")
    )

    # Join to nat bounds
    sf_slopes_ur <- nat_bounds %>%
        select(WB_NAME, SUBREGION, geometry) %>%
        right_join( slopes_ur, by = c("WB_NAME" = "country") )

    sf_slopes_pw_avg <- nat_bounds |> 
        select(WB_NAME, SUBREGION, geometry) %>%
        right_join( slopes_pw_avg, by = c("WB_NAME" = "country") )

    list(
        "slopes_ur"     = slopes_ur,
        "slopes_pw_avg" = slopes_pw_avg,
        "sf_slopes_ur"  = sf_slopes_ur,
        "sf_slopes_pw_avg"  = sf_slopes_pw_avg,
        "pc1_draws_ur"  = pc1_draws_ur,
        "pc1_draws_pw_avg" = pc1_draws_pw_avg
    )
}

#### Models for 2000-2022:

In [6]:
# Prepare data
DATA_PATH   <- file.path(SCRATCH_DIR, 
                         "data/spatial/fire_pm_dep_paper_data",
                         "proc_data/df_af_annual_loc_tp_2000_2023.csv")

df <- read_csv(DATA_PATH) |> 
    filter(year <= 2022) |> 
    group_by(lon, lat) |>
    mutate(grid_id = cur_group_id()) |> # create grid_id for projecting SE vars
    ungroup()

df <- project_indicators(
    df, 
    list(edu_mean_years     = 2017,
         imp_san_access_pct = 2017,
         stunting_pct_u5    = 2017
    )
)

# Do PCA
pca_res <- compute_pca(
    df,
    method          = "ppca",
    se_indicators   = c("edu_mean_years", "imp_san_access_pct", "log_GDP_pc",
                        "child_dep_pct", "stunting_pct_u5"),
    n_pcs           = 5,
    scale           = TRUE,
    centre          = TRUE,
    seed            = 42,
    positive_vars   = c("child_dep_pct", "stunting_pct_u5"),
    negative_vars   = c("edu_mean_years", "imp_san_access_pct", "log_GDP_pc")
)

# Join PCs back onto df (helper funct)
df  <- prepare_analysis_data(df, 
                             pca_res$data, 
                             outcome = "fire_PM25_hu", # arbitrary, won't be used
                             scale_y = FALSE,
                             scale_PC1 = TRUE)

Rows: 994412 Columns: 107
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (11): geometry, ISO_A3, country, region, continent, INCOME_GRP, ECONOMY,...
dbl (96): lon, lat, total_PM25, total_O3, fire_PM25, fire_O3, fire_PM25_hu, ...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


[1] "Projecting edu_mean_years forward from 2017"


Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 0.0227167 (tol = 0.002, component 1)”
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?”


[1] "Projecting imp_san_access_pct forward from 2017"


Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 0.00349425 (tol = 0.002, component 1)”


[1] "Projecting stunting_pct_u5 forward from 2017"


Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 0.0278442 (tol = 0.002, component 1)”
Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?”
Warning message:
“Using an external vector in selections was deprecated in tidyselect 1.1.0.
ℹ Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(se_indicators)

  # Now:
  data %>% select(all_of(se_indicators))

See <https://tidyselect.r-lib.org/reference/faq-external-vector.html>.”
Warning message in pcaMethods::ppca(X, nPcs = n_pcs, seed = seed):
“stopped after max iterations, but rel_ch was > threshold”


In [7]:
path_loc <- file.path(
    SCRATCH_DIR, paste0(
        "data/spatial/deprivation_pm_bhm/",
        "original_model_hu_bc_attr_fire_PM25_emis_0_100km_scaleyFALSE_ppcaPcaMethod_5pcs",
        "_4chains_15000iter_5000warmup_24threads_normal10_betaPrior_cauchy_scalePrior.rds"
    )
)

path_tp <- file.path(
    SCRATCH_DIR, paste0(
        "data/spatial/deprivation_pm_bhm/",
        "original_model_hu_bc_attr_fire_PM25_emis_100_2000km_scaleyFALSE_ppcaPcaMethod_5pcs",
        "_4chains_12000iter_4000warmup_24threads_normal10_betaPrior_cauchy_scalePrior.rds"
    )
)

In [8]:
# Load models
model_loc <- readRDS(path_loc)
model_tp  <- readRDS(path_tp)

In [12]:
# Post-process models
slopes_model_loc    <- postprocess(model_loc, df, nat_bounds)
slopes_model_tp     <- postprocess(model_tp, df, nat_bounds)

Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”


##### “More deprived areas experienced higher exposure to local fire pollution in X countries, compared to Y countries where deprivation was associated with lower local pollution exposure”

In [13]:
slopes_model_loc$slopes_pw_avg |> 
    mutate(
        direction = case_when(
            conf.low > 0    ~ "positive",
            conf.high < 0   ~ "negative",
            conf.low <= 0 & conf.high >= 0  ~ "null",
            .default = NA_character_
        )
    ) |> count(direction)

direction,n
<chr>,<int>
negative,8
null,22
positive,23


##### “More deprived areas had higher exposure to transported fire pollution in X countries, but lower exposure in Y countries”

In [14]:
slopes_model_tp$slopes_pw_avg |> 
    mutate(
        direction = case_when(
            conf.low > 0    ~ "positive",
            conf.high < 0   ~ "negative",
            conf.low <= 0 & conf.high >= 0  ~ "null",
            .default = NA_character_
        )
    ) |> count(direction)

direction,n
<chr>,<int>
negative,15
null,16
positive,22


##### “in Burundi, Cameroon, DR Congo, Ghana, Malawi, Uganda, more deprived areas experienced higher local fire pollution while more affluent areas were exposed to higher transported pollution”

[[Countries with positive deprivation effect for local but negative effect for transported]]

In [15]:
intersect(
    x = slopes_model_loc$slopes_pw_avg |> 
            filter(conf.low > 0 ) |> 
            pull(country),
    y = slopes_model_tp$slopes_pw_avg |> 
            filter(conf.high < 0) |> 
            pull(country)
)

[1] "Burundi"                       "Cameroon"                     
[3] "Congo, Democratic Republic of" "Ghana"                        
[5] "Malawi"                        "Uganda"

#### Models for 2000-2017:

In [6]:
# Prepare data
DATA_PATH   <- file.path(SCRATCH_DIR, 
                         "data/spatial/fire_pm_dep_paper_data",
                         "proc_data/df_af_annual_loc_tp.csv")

df <- read_csv(DATA_PATH) |> 
    filter(year <= 2017) |> 
    group_by(lon, lat) |>
    mutate(grid_id = cur_group_id()) |> # create grid_id for projecting SE vars
    ungroup()

# Do PCA
pca_res <- compute_pca(
    df,
    method          = "ppca",
    se_indicators   = c("edu_mean_years", "imp_san_access_pct", "log_GDP_pc",
                        "child_dep_pct", "stunting_pct_u5"),
    n_pcs           = 5,
    scale           = TRUE,
    centre          = TRUE,
    seed            = 42,
    positive_vars   = c("child_dep_pct", "stunting_pct_u5"),
    negative_vars   = c("edu_mean_years", "imp_san_access_pct", "log_GDP_pc")
)

# Join PCs back onto df (helper funct)
df  <- prepare_analysis_data(df, 
                             pca_res$data, 
                             outcome = "fire_PM25_hu", # arbitrary, won't be used
                             scale_y = FALSE,
                             scale_PC1 = TRUE)

Rows: 828600 Columns: 102
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (11): geometry, ISO_A3, country, region, continent, INCOME_GRP, ECONOMY,...
dbl (91): lon, lat, total_PM25, total_O3, fire_PM25, fire_O3, year, GDP_MD_E...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Warning message:
“Using an external vector in selections was deprecated in tidyselect 1.1.0.
ℹ Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(se_indicators)

  # Now:
  data %>% select(all_of(se_indicators))

See <https://tidyselect.r-lib.org/reference/faq-external-vector.html>.”


In [10]:
path_loc <- file.path(
    SCRATCH_DIR, paste0(
        "data/spatial/deprivation_pm_bhm/",
        "original_model_hu_bc_attr_fire_PM25_emis_0_100km_scaleyFALSE_ppcaPcaMethod_5pcs_maxYear2017",
        "_4chains_15000iter_5000warmup_24threads_normal10_betaPrior_cauchy_scalePrior.rds"
    )
)

path_tp <- file.path(
    SCRATCH_DIR, paste0(
        "data/spatial/deprivation_pm_bhm/",
        "original_model_hu_bc_attr_fire_PM25_emis_100_2000km_scaleyFALSE_ppcaPcaMethod_5pcs_maxYear2017",
        "_4chains_15000iter_5000warmup_24threads_normal10_betaPrior_cauchy_scalePrior.rds"
    )
)

In [12]:
# Load models
model_loc <- readRDS(path_loc)
model_tp  <- readRDS(path_tp)

In [13]:
# Post-process models
slopes_model_loc    <- postprocess(model_loc, df, nat_bounds)
slopes_model_tp     <- postprocess(model_tp, df, nat_bounds)

Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”
Warning message:
“Dropping 'draws_df' class as required metadata was removed.”


##### “More deprived areas experienced higher exposure to local fire pollution in X countries, compared to Y countries where deprivation was associated with lower local pollution exposure”

In [14]:
slopes_model_loc$slopes_pw_avg |> 
    mutate(
        direction = case_when(
            conf.low > 0    ~ "positive",
            conf.high < 0   ~ "negative",
            conf.low <= 0 & conf.high >= 0  ~ "null",
            .default = NA_character_
        )
    ) |> count(direction)

direction,n
<chr>,<int>
negative,9
null,22
positive,22


##### “More deprived areas had higher exposure to transported fire pollution in X countries, but lower exposure in Y countries”

In [15]:
slopes_model_tp$slopes_pw_avg |> 
    mutate(
        direction = case_when(
            conf.low > 0    ~ "positive",
            conf.high < 0   ~ "negative",
            conf.low <= 0 & conf.high >= 0  ~ "null",
            .default = NA_character_
        )
    ) |> count(direction)

direction,n
<chr>,<int>
negative,12
null,23
positive,18


##### “in Burundi, Cameroon, DR Congo, Ghana, Malawi, Uganda, more deprived areas experienced higher local fire pollution while more affluent areas were exposed to higher transported pollution”

[[Countries with positive deprivation effect for local but negative effect for transported]]

In [16]:
intersect(
    x = slopes_model_loc$slopes_pw_avg |> 
            filter(conf.low > 0 ) |> 
            pull(country),
    y = slopes_model_tp$slopes_pw_avg |> 
            filter(conf.high < 0) |> 
            pull(country)
)

[1] "Burundi"                       "Cameroon"                     
[3] "Congo, Democratic Republic of" "Ghana"                        
[5] "Malawi"